# Results for model: Qwen/Qwen2.5-72B-Instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load data
data = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

# Feature Engineering
def calculate_top_quantile(df):
    df = df.with_column(
        pl.col('feature_00').quantile(0.85).over('date_id').alias('top_quantile')
    )
    return df

data = data.groupby_rolling(index_column='date_id', period='1i').apply(calculate_top_quantile)

# Prepare data for training
X = data.select(pl.exclude(['responder_6', 'date_id']))
y = data.select('responder_6')

# Convert to DMatrix
dtrain = xgb.DMatrix(X.to_pandas(), label=y.to_pandas())

# Train XGBoost Regressor
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'eta': 0.1
}
model = xgb.train(params, dtrain, num_boost_round=100)

# Save model
model.save_model('jane_street_model.json')